# Heat Content in the Mixed Layer

This example demonstrates computing vertically integrated heat content (per unit area) within the mixed layer, following a similar flow to the Argo examples.

It:
- Loads a small set of Argo profiles,
- Computes Conservative Temperature (CT),
- Builds a depth grid and thicknesses using hydrostatic balance,
- Estimates the mixed layer depth via a simple temperature threshold, and
- Integrates heat content down to the mixed layer using the new `heat_content` utility.

Note: You will need `gsw`, `xarray`, and `netCDF4` installed, and a local copy of Argo netCDF files. Update the `Path_to_Argo` below to point to your data.

In [ ]:
import numpy as np
import xarray as xr
import gsw
from glob import glob

from oceanmixedlayers import oceanmixedlayers

%load_ext autoreload
%autoreload 2

In [ ]:
def IsVarThere(hndl, VAR):
    return VAR in hndl.data_vars

def good_qc_flags(qc_arr):
    # Return boolean mask where QC is in {1,2,5,8}
    qc = np.array(qc_arr, dtype=float)
    return (qc==1)|(qc==2)|(qc==5)|(qc==8)

In [ ]:
# Point this path at your local Argo dataset (DAC folders with *_prof.nc files)
Path_to_Argo = '/net3/bgr/Datasets/Argo/202011-ArgoData/'  # update as needed

# For a quick demo, grab a handful of profiles from one DAC/platform
profile_paths = sorted(glob(Path_to_Argo + 'dac/*/*/*_prof.nc'))
print('Found', len(profile_paths), 'profile files')

oml = oceanmixedlayers()

In [ ]:
def process_file(path, max_profiles=20, temp_threshold=0.2, ref_depth=-10.0, T_ref=0.0):
    """
    Parse a single Argo *_prof.nc, compute MLD by a temperature threshold,
    and heat content down to that MLD using CT.

    temp_threshold: delta T [deg C] from near-surface used for MLD.
    ref_depth: reference depth [m, negative downward] used for surface value (default -10 m).
    T_ref: reference temperature for heat content [deg C].
    """
    out = []
    with xr.open_dataset(path) as ds:
        if not (IsVarThere(ds, 'TEMP') and IsVarThere(ds, 'PSAL') and IsVarThere(ds, 'PRES')):
            return []
        NP = int(ds.N_PROF)
        take = min(NP, max_profiles)
        for p in range(take):
            day_qc = float(ds.JULD_QC[p])
            pos_qc = float(ds.POSITION_QC[p])
            if not (((day_qc==1) or (day_qc==2) or (day_qc==5) or (day_qc==8))
                    and ((pos_qc==1) or (pos_qc==2) or (pos_qc==5) or (pos_qc==8))):
                continue
            PRES = np.array(ds.PRES[p,:], dtype=float)
            if np.all(np.isnan(PRES)):
                continue
            SALTQC = np.array(ds.PSAL_QC[p,:], dtype=float)
            TEMPQC = np.array(ds.TEMP_QC[p,:], dtype=float)
            PRESQC = np.array(ds.PRES_QC[p,:], dtype=float)
            LI = good_qc_flags(SALTQC) & good_qc_flags(TEMPQC) & good_qc_flags(PRESQC)
            if np.sum(LI) < 20:
                continue
            P = np.array(ds.PRES[p,:].values[LI], dtype=float)  # dbar
            S = np.array(ds.PSAL[p,:].values[LI], dtype=float)
            T = np.array(ds.TEMP[p,:].values[LI], dtype=float)
            # QC sanity checks
            QQ = ((np.min(P) >= 0) and (np.max(P) < 3000)
                  and (np.min(S) > 0) and (np.max(S) < 50)
                  and (np.min(T) > -2) and (np.max(T) < 50))
            monotonic = np.all(P[:-1] < P[1:])
            if not (QQ and monotonic):
                continue
            # Compute CT (deg C)
            CT = gsw.conversions.CT_from_t(S, T, P)
            # Build layer interfaces with a surface level at 0 dbar
            P_i = np.concatenate(([0.0], P))            # dbar
            CT_i = np.concatenate(([CT[0]], CT))        # deg C
            S_i = np.concatenate(([S[0]], S))
            # Layer means
            P_c = 0.5*(P_i[1:] + P_i[:-1])
            CT_c = 0.5*(CT_i[1:] + CT_i[:-1])
            # Estimate in-situ density for hydrostatic dz (use CT_c, P_c)
            rho = gsw.density.rho(S, CT_c, P_c/1.0)   # kg/m^3, P in dbar -> gsw expects dbar
            dP = P_i[:-1] - P_i[1:]                  # negative (Pa/dbar not yet)
            # Convert dbar to Pa, then dz via hydrostatic balance
            g = 9.81
            dZ = 1.0e4 * dP / (g * rho)              # meters, positive
            Z_i = np.zeros_like(P_i)
            for zi in range(len(rho)):
                Z_i[zi+1] = Z_i[zi] + 1.0e4 * (P_i[zi] - P_i[zi+1]) / (g * rho[zi])
            Z_c = 0.5*(Z_i[1:] + Z_i[:-1]) * -1.0     # negative downward
            dZ = (Z_i[:-1] - Z_i[1:])                # positive thickness
            # Mixed layer depth via temperature threshold (ref at ~-10 m)
            # Find index nearest to ref_depth
            ref_idx = np.argmin(np.abs(Z_c - ref_depth))
            Tref = CT_c[ref_idx]
            # Find first depth where |CT - Tref| > threshold
            over = np.where(np.abs(CT_c - Tref) > temp_threshold)[0]
            if len(over) == 0:
                # no threshold crossing; skip profile
                continue
            k = over[0]
            mld = -Z_c[k]  # approximate MLD in meters (positive)
            # Heat content to MLD (negative depth)
            hc = oml.heat_content(Z_c, dZ, CT_c, depth=-mld, T_ref=T_ref)
            out.append({
                'mld_m': mld,
                'heat_content_Jpm2': float(hc),
                'latitude': float(ds.LATITUDE[p].values) if 'LATITUDE' in ds else np.nan,
                'longitude': float(ds.LONGITUDE[p].values) if 'LONGITUDE' in ds else np.nan,
                'julian_day': float(ds.JULD[p].values) if 'JULD' in ds else np.nan,
                'file': path
            })
    return out

In [ ]:
# Run processing on a small sample of files
records = []
for f in profile_paths[:2]:  # limit to a couple of files for speed
    recs = process_file(f, max_profiles=25, temp_threshold=0.2, ref_depth=-10.0, T_ref=0.0)
    records.extend(recs)
print('Processed records:', len(records))

import pandas as pd
df = pd.DataFrame.from_records(records)
df.head()

In [ ]:
# Simple scatter of heat content vs MLD (if matplotlib available)
import matplotlib.pyplot as plt

if len(records) > 0:
    plt.figure(figsize=(5,4))
    plt.scatter(df['mld_m'], df['heat_content_Jpm2'], s=10, alpha=0.7)
    plt.xlabel('MLD [m]')
    plt.ylabel('Heat Content [J/m^2]')
    plt.title('Mixed Layer Heat Content (threshold MLD)')
    plt.grid(True)
    plt.show()
else:
    print('No records to plot; check Path_to_Argo and QC filters.')